# 15 · Tablero territorial integrado

**Objetivo:** Combinar vegetación, agua, cobertura y precipitación en un mapa único.

**Datos:** Sentinel-2, Dynamic World y CHIRPS.

**Relevancia para política ambiental y social:** Presenta indicadores sintéticos para diálogo de política pública.

**Limitaciones:** Los indicadores deben interpretarse junto con evidencia social y de campo.


In [ ]:
# Instalar dependencias en Google Colab
!pip -q install earthengine-api geemap

import ee
import geemap
import datetime

ee.Authenticate()
ee.Initialize(project="TU_PROYECTO_GEE")

# Área de estudio de ejemplo: entorno de Chachapoyas, Amazonas, Perú
# Reemplázala por un polígono, activo de Earth Engine o coordenadas propias.
aoi = ee.Geometry.Point([-77.87, -6.23]).buffer(30000)

Map = geemap.Map()
Map.centerObject(aoi, 9)


In [ ]:
def mask_s2_sr(image):
    scl = image.select("SCL")
    clear = (
        scl.neq(3)   # sombra
        .And(scl.neq(8))  # nube media
        .And(scl.neq(9))  # nube alta
        .And(scl.neq(10)) # cirrus
        .And(scl.neq(11)) # nieve/hielo
    )
    return image.updateMask(clear).divide(10000).copyProperties(
        image, ["system:time_start"]
    )

def s2_composite(start, end):
    return (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(aoi)
        .filterDate(start, end)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 40))
        .map(mask_s2_sr)
        .median()
        .clip(aoi)
    )


In [ ]:
s2 = s2_composite("2025-01-01","2025-12-31")
ndvi = s2.normalizedDifference(["B8","B4"]).rename("NDVI")
mndwi = s2.normalizedDifference(["B3","B11"]).rename("MNDWI")
landcover = ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1").filterBounds(aoi).filterDate("2025-01-01","2025-12-31").select("label").mode().clip(aoi)
rain = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY").filterBounds(aoi).filterDate("2025-01-01","2025-12-31").sum().clip(aoi)

Map.addLayer(s2, {"bands":["B4","B3","B2"],"min":0,"max":0.3}, "RGB", False)
Map.addLayer(ndvi, {"min":0,"max":0.9,"palette":["brown","yellow","green"]}, "NDVI")
Map.addLayer(mndwi.gt(0.1).selfMask(), {"palette":["cyan"]}, "Agua")
Map.addLayer(landcover, {"min":0,"max":8,"palette":["419BDF","397D49","88B053","7A87C6","E49635","DFC35A","C4281B","A59B8F","B39FE1"]}, "Cobertura", False)
Map.addLayer(rain, {"min":500,"max":3000,"palette":["yellow","green","blue"]}, "Precipitación", False)

summary = ee.Image.cat([ndvi, mndwi, rain.rename("rain")]).reduceRegion(
    ee.Reducer.mean(), aoi, 100, maxPixels=1e9
)
print("Indicadores medios:", summary.getInfo())
Map
